**IMPORTACION DEL LIBRERIAS**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Librerías cargadas con éxito.")

**CARGA Y EXPLORACION DE LOS DATOS**

In [ ]:
# Cargar el archivo adjunto (ajusta la ruta según dónde guardaste el CSV)
# Cargar el archivo procesado usando la ruta relativa correcta
df = pd.read_csv('../data/processed/bank_processed.csv')

print(f"Dimensiones del dataset: {df.shape}")
print("\nDistribución de la variable objetivo (deposit):")
print(df['deposit'].value_counts(normalize=True))

# Mostrar las primeras filas
df.head()

**PRE-PROCESAMIENTO**
conversion de categoricos a numericos rapidos

In [ ]:
# Identificar columnas categóricas de texto para convertirlas a números (One-Hot Encoding)
categorical_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']

# Aplicar codificación
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separar características (X) y etiqueta objetivo (y)
X = df_encoded.drop(columns=['deposit'])
y = df_encoded['deposit']

print(f"Características listas para entrenar: {X.shape[1]} columnas.")

**DIVISION DE DATOS (ENTRENAMIENTO Y PRUEBA)**

In [ ]:
# Dividir el dataset: 80% para entrenar, 20% para evaluar el rendimiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Muestras de entrenamiento: {X_train.shape[0]}")
print(f"Muestras de prueba: {X_test.shape[0]}")

**DEFINICION Y ENTRENAMIENTO DEL MODELO**

In [ ]:
# Inicializar el modelo con los hiperparámetros del diseño institucional
model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42
)

# Entrenar el modelo con los datos de entrenamiento
model.fit(X_train, y_train)
print("¡Modelo Random Forest entrenado exitosamente!")

**EVALUACION DEL MODELO**

In [ ]:
# Realizar predicciones sobre el conjunto de pruebas
y_pred = model.predict(X_test)

# Calcular exactitud base
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud del modelo (Accuracy): {accuracy:.4f}\n")

# Mostrar reporte de métricas detallado (Precisión, Recall, F1-Score)
print("--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

# Graficar la matriz de confusión para auditoría visual
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Acepta', 'Acepta'], yticklabels=['No Acepta', 'Acepta'])
plt.title('Matriz de Confusión - Caso Bank Marketing')
plt.xlabel('Predicción del Modelo')
plt.ylabel('Clase Real (Destino)')
plt.show()

**GRAFICOS PARA LEER MEJOR LOS RESULTADOS**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Configuración estética general para los gráficos
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})

# Crear un lienzo con 3 subgráficos dispuestos de forma ordenada
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ----------------------------------------------------
# GRÁFICO 1: Matriz de Confusión Explicada
# ----------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
# Calcular porcentajes para anotarlos dentro de la matriz
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Crear las etiquetas de texto combinando cantidad exacta y porcentaje
labels = [f"{v1}\n({v2:.1f}%)" for v1, v2 in zip(cm.flatten(), cm_percent.flatten())]
labels = np.asarray(labels).reshape(2,2)

sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=['No Contrata', 'Sí Contrata'], 
            yticklabels=['No Contrata', 'Sí Contrata'])
axes[0].set_title('¿Dónde acierta y dónde falla el modelo?\n(Matriz de Confusión)')
axes[0].set_xlabel('Predicción de la IA')
axes[0].set_ylabel('Realidad del Cliente')

# ----------------------------------------------------
# GRÁFICO 2: Comparativa Visual de Métricas (F1, Precision, Recall)
# ----------------------------------------------------
# Extraemos el diccionario del reporte de clasificación para graficarlo
report_dict = classification_report(y_test, y_pred, output_dict=True)
metrics_df = pd.DataFrame(report_dict).iloc[:-1, :2].T # Filtrar solo clases 0 y 1

metrics_melted = metrics_df.reset_index().melt(id_vars='index', var_name='Métrica', value_name='Valor')
metrics_melted.columns = ['Clase', 'Métrica', 'Valor']
metrics_melted['Clase'] = metrics_melted['Clase'].map({'0': 'No Contrata', '1': 'Sí Contrata'})

sns.barplot(data=metrics_melted, x='Métrica', y='Valor', hue='Clase', palette='Set2', ax=axes[1])
axes[1].set_title('Rendimiento por Objetivo de Negocio')
axes[1].set_xlabel('Métricas de Evaluación')
axes[1].set_ylabel('Puntaje (0 a 1)')
axes[1].set_ylim(0, 1.1)
for p in axes[1].patches:
    if p.get_height() > 0:
        axes[1].annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height() + 0.02),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=10)

# ----------------------------------------------------
# GRÁFICO 3: ¿Qué es lo que más le importa al Modelo? (Feature Importance)
# ----------------------------------------------------
# Extraer la importancia de las variables del Random Forest
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

# Tomar el Top 10 de las variables que más influyen
top_n = 10
features_df = pd.DataFrame({
    'Variable': [X.columns[i] for i in indices[:top_n]],
    'Importancia': importances[indices[:top_n]]
})

sns.barplot(data=features_df, x='Importancia', y='Variable', palette='viridis', ax=axes[2])
axes[2].set_title('Top 10 Factores Críticos\n(¿Qué influye más en el cliente?)')
axes[2].set_xlabel('Nivel de Importancia Relativa')
axes[2].set_ylabel('')

# Ajustar diseño final y mostrar
plt.tight_layout()
plt.show()